# The quantum eraser — decoherence is bookkeeping, not damage

**The punchline.** Decohere a qubit completely: its superposition is gone, its entropy is
a full bit, its interference is a fair coin. Then run the coupling **backwards**, and
every bit of it comes back — exactly, to floating-point precision. Nothing was destroyed,
because nothing was ever thrown away. Decoherence is a fact about which parts of the world
you are keeping track of, not a fact about the state.

And then the honest half: measure the environment first, and the same undo does nothing.
The difference between those two cells is the difference between a *coherent record* and
a *classical outcome*.

Background: **[06 — Why the world looks classical](../06-decoherence.ipynb)** §9, and
**[04 — Programs made of gates](../04-combinators.ipynb)** §7 for the tape.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, QsimError
from qsim.decoherence import dephasing_coupling
from qsim.gates import H

## 1. Kill the superposition

$\theta = \pi$ is the full setting of the dial from
[decoherence_dial](decoherence_dial.ipynb): the environment qubit ends up in
$\lvert 0\rangle$ or $\lvert 1\rangle$ exactly according to the system, a perfect
which-path record.

In [ ]:
qc = Circuit(name="eraser", seed=99)
q = qc.alloc("q")
env = qc.environment_qubit()

H(q)
print(f"fresh |+>       :  coherence {qc.inspect.coherence(q):.12f}"
      f"   system entropy {qc.inspect.system_entropy():.12f} bits")

dephasing_coupling(q, env, theta=np.pi)
print(f"after coupling  :  coherence {qc.inspect.coherence(q):.12f}"
      f"   system entropy {qc.inspect.system_entropy():.12f} bits")
print()
print("the system's density matrix — a classical coin flip, no coherence left:")
print(np.round(qc.inspect.system_density_matrix(), 12))
print()
print("...while the *whole* state is as pure as it ever was:")
print("   ", qc.inspect.ket())
print("    entropy of the pair:",
      round(qc.inspect.entanglement_entropy(list(qc.qubits)), 12), "bits")

Those last two lines are the whole notebook in advance. The system looks like a coin
flip. The system-plus-environment is $(\lvert 00\rangle + \lvert 11\rangle)/\sqrt2$ — a
Bell state, pure, zero entropy, complete information. The coin flip is an artefact of
where we drew the line.

## 2. Un-kill it

`dephasing_coupling` is a `@qsim.gate` **block**, so it has an adjoint the same way a
single gate does: `dephasing_coupling.adjoint()` is another block, named
`dephasing_coupling†`, that runs the same ops reversed and inverted. Applying it undoes
the record.

In [ ]:
dephasing_coupling.adjoint()(q, env, theta=np.pi)

print(f"after erasure   :  coherence {qc.inspect.coherence(q):.12f}"
      f"   system entropy {qc.inspect.system_entropy():.12f} bits")

H(q)   # close the interferometer: if the coherence really came back, this is certainty
print(f"\nfinal P(0) = {qc.inspect.probabilities()[0]:.12f}")
print("blocks this circuit is made of:", qc.block_counts())
print("gates it actually cost        :", qc.gate_counts())

Coherence $0.5 \to 0 \to 0.5$; entropy $0 \to 1 \to 0$; and the interferometer closes on
certainty. Not "approximately", not "on average over many runs" — exactly.

Look at the last two lines. The tape records the erasure as *work done*, not as a
deletion: `dephasing_coupling` and `dephasing_coupling†` both appear, and the gate count
includes both. That is deliberate. Undoing decoherence costs real operations on real
hardware, and a circuit diagram that hid them would be a diagram of a program nobody ran.

## 3. Watching it happen, one gate at a time

`qc.on_op(fn)` attaches an observer that is called after every operation, with the state
already updated. It is the way to watch a program without editing the program.

Below, the same journey is taken in twelve small steps instead of two big ones: six
couplings of $\pi/6$, then six inverse couplings. Rotations about the same axis add, so
six lots of $\pi/6$ *is* the $\pi$ coupling above — which means the walk down and the
walk back up are visible in between.

In [ ]:
traced = Circuit(name="traced-eraser", seed=99)
tq = traced.alloc("q")
tenv = traced.environment_qubit()

coherence_trace: list[float] = []
entropy_trace: list[float] = []
op_names: list[str] = []


def watch(op, circuit) -> None:
    """Called after every op the circuit runs. Hooks observe; they may not emit."""
    coherence_trace.append(circuit.inspect.coherence(tq))
    entropy_trace.append(circuit.inspect.system_entropy())
    op_names.append(op.name)


handle = traced.on_op(watch)

H(tq)                                                    # open the interferometer
for _ in range(6):
    dephasing_coupling(tq, tenv, theta=np.pi / 6)      # let the record build up
for _ in range(6):
    dephasing_coupling.adjoint()(tq, tenv, theta=np.pi / 6)   # and unbuild it
H(tq)                                                    # close the interferometer

handle.remove()

steps = np.arange(1, len(coherence_trace) + 1)
fig, ax = plt.subplots(figsize=(9.0, 3.8))
ax.plot(steps, np.array(coherence_trace) * 2.0, marker="o", lw=2, color="crimson",
        label=r"coherence $2\,|\rho_{01}|$")
ax.plot(steps, entropy_trace, marker="s", lw=2, color="purple",
        label="system entropy (bits)")
ax.axvspan(1.5, 7.5, color="gray", alpha=0.12)
ax.text(4.5, 1.16, "recording", ha="center", color="gray")
ax.text(10.5, 1.16, "erasing", ha="center", color="gray")
ax.set_xticks(steps)
ax.set_xticklabels(op_names, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("op, in the order the circuit ran them")
ax.set_ylim(-0.05, 1.3)
ax.legend(fontsize=9, loc="center left")
ax.set_title("one measurement of the world per gate, taken by a hook")
fig.tight_layout()

print(f"final P(0) = {traced.inspect.probabilities()[0]:.12f}")

A perfect V. The coherence walks down to zero as the environment accumulates the record
and walks back up as it is unwound; the entropy does the mirror image. Every point on the
way down has a matching point on the way up — the eraser is not a special operation, it
is the recording played backwards.

Note the op names on the axis: `Ry` all the way across. The recording and the erasing are
*the same kind of gate*. There is no operation in this library called "decohere" and none
called "erase". There is a rotation, and a rotation by the negative angle.

## 4. The honest half: measure the environment first

Everything above depends on the record being **coherent** — written by a gate, still on
the tape, still invertible. Replace the gate with a measurement and the situation changes
completely, and not because the library declines to try.

In [ ]:
severed = Circuit(name="severed", seed=99)
sq = severed.alloc("q")
senv = severed.environment_qubit()

H(sq)
mark = severed.checkpoint()          # a position on the tape, taken before the recording
dephasing_coupling(sq, senv, theta=np.pi)

reading = severed.measure(senv)   # the environment is *read*, not merely correlated
print(f"the environment was measured and said: {reading}")

dephasing_coupling.adjoint()(sq, senv, theta=np.pi)   # try to erase anyway
coherence_after_failed_erasure = severed.inspect.coherence(sq)
print(f"after the same erasure:  coherence {coherence_after_failed_erasure:.12f}")

H(sq)
# Entry [0, 0] of q's own density matrix is P(q = 0), with the environment ignored.
severed_p0 = float(np.real(severed.inspect.reduced_density_matrix([sq])[0, 0]))
print(f"final P(q = 0) = {severed_p0:.12f}   (0.5 = no interference left)")

Nothing came back. The undo ran — it is a perfectly good unitary — and it had nothing to
undo, because the measurement had already discarded the branch it did not report. There
is no operation that brings a discarded branch back.

The tape says the same thing in words. `rewind` walks the recorded ops backwards, and it
refuses to walk through a measurement:

In [ ]:
try:
    severed.rewind(mark)
except QsimError as err:
    print(err)

## 5. So why is the laboratory not like section 2?

Nothing in the physics prevents the eraser. What prevents it in practice is size.

A real qubit does not decohere against *one* other qubit; it decoheres against the
$10^{20}$-odd photons and air molecules that scatter off it, each carrying a fragment of
the record away at the speed of light in a different direction. Undoing that would mean
catching every fragment and applying the exact inverse rotation to each — a unitary that
exists, that no one will ever build, and whose difficulty grows with every microsecond
that passes.

That is **FAPP** irreversibility — "for all practical purposes", John Bell's phrase, used
by him as a complaint. It is a statement about engineering budgets, not about physics, and
it is worth keeping the two apart. This notebook is the physics; the laboratory is the
budget.

There is one case where the distinction may be more than practical: a record carried past
a horizon — the event horizon of a black hole, or the cosmological horizon — is one no
observer inside can ever retrieve, even in principle. `qsim`'s planned `horizon.ipynb`
demo makes that the difference between an operation that returns a wrong answer and one
that *raises*: the API distinction is the philosophical distinction. It is not built yet,
because the marking it needs is not built yet.

## Where to go next

- **[decoherence_dial](decoherence_dial.ipynb)**: the partial settings between the two
  extremes here, and the $V^2 + D^2 = 1$ budget.
- **[wigners_friend](wigners_friend.ipynb)**: the same erasure with the environment qubit
  called an observer, which turns section 4 into a puzzle about measurement.
- **[04 — Programs made of gates](../04-combinators.ipynb)** §7: `checkpoint`, `rewind`,
  and why the tape is never rewritten.

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. Full decoherence, then exact recovery — the central claim.
check = Circuit(name="assert-eraser", seed=1)
cq = check.alloc("q")
cenv = check.environment_qubit()
H(cq)
assert np.isclose(check.inspect.coherence(cq), 0.5)

dephasing_coupling(cq, cenv, theta=np.pi)
assert np.isclose(check.inspect.coherence(cq), 0.0, atol=1e-12)
assert np.isclose(check.inspect.system_entropy(), 1.0)

dephasing_coupling.adjoint()(cq, cenv, theta=np.pi)
assert np.isclose(check.inspect.coherence(cq), 0.5, atol=1e-12)
assert np.isclose(check.inspect.system_entropy(), 0.0, atol=1e-12)

H(cq)
assert np.isclose(check.inspect.probabilities()[0], 1.0, atol=1e-12)

# 2. The whole state was pure at every moment, including the decohered one.
assert np.isclose(check.inspect.entanglement_entropy(list(check.qubits)), 0.0, atol=1e-12)

# 3. The tape records the erasure as work, not as a deletion.
assert qc.block_counts() == {"dephasing_coupling": 1, "dephasing_coupling†": 1}

# 4. The traced version walks down and back up symmetrically, and closes on certainty.
assert np.isclose(min(coherence_trace), 0.0, atol=1e-12)
assert np.isclose(max(entropy_trace), 1.0)
assert np.allclose(coherence_trace[1:7], coherence_trace[11:5:-1], atol=1e-12)
assert np.isclose(coherence_trace[0], coherence_trace[12], atol=1e-12)
assert np.isclose(traced.inspect.probabilities()[0], 1.0, atol=1e-12)

# 5. Once the environment is measured, the same erasure recovers nothing.
assert np.isclose(coherence_after_failed_erasure, 0.0, atol=1e-12)
assert np.isclose(severed_p0, 0.5, atol=1e-12)

print("all assertions passed")